# MA3627 — Workshop 7: Decision Trees and Ensemble Methods

This workshop accompanies the Week 7 lecture. Parts A–D build, visualise, and prune a
single decision tree, and demonstrate its instability. Part E studies feature importance
and its known bias. Parts F–I build the three ensemble families — bagging, Random
Forests, and gradient boosting — directly on top of that instability.

Take-home exercises are at the end.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from scipy.stats import mode
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(0)

# Digits: 1797 samples, 64 pixel-intensity features (8x8 images), 10 classes.
# Used throughout this workshop for both the tree and the ensemble parts.
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dig, y_dig, test_size=0.3, random_state=0, stratify=y_dig
)
print(f"Digits: {X_dig.shape[0]} samples, {X_dig.shape[1]} features, {len(np.unique(y_dig))} classes")
print(f"Train / test: {len(y_tr)} / {len(y_te)}")

---
## Part A — Impurity measures and the greedy split from scratch

We implement Gini impurity and entropy, reproduce the lecture's numerical example, and
build a single-level greedy best-split search.

In [ ]:
def gini(y):
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y, minlength=y.max() + 1)
    p = counts / n
    return 1.0 - np.sum(p ** 2)

def entropy(y):
    n = len(y)
    if n == 0:
        return 0.0
    counts = np.bincount(y, minlength=y.max() + 1)
    p = counts[counts > 0] / n
    return -np.sum(p * np.log(p))

print("{:<35} {:>8} {:>8}".format("Distribution", "Gini", "H/ln2"))
print("-" * 53)
cases = [
    ("Pure node (all class 0)",       np.array([0, 0, 0, 0, 0])),
    ("Binary, balanced (2 classes)",  np.array([0, 0, 1, 1])),
    ("Ternary, balanced (3 classes)", np.array([0, 0, 1, 1, 2, 2])),
    ("Uniform over 10 classes",       np.repeat(np.arange(10), 5)),
]
for label, y in cases:
    g = gini(y)
    h = entropy(y) / np.log(2)
    print(f"{label:<35} {g:>8.4f} {h:>8.4f}")

In [ ]:
def impurity_reduction(y, y_left, y_right, criterion="gini"):
    fn = gini if criterion == "gini" else entropy
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    return fn(y) - (n_l / n) * fn(y_left) - (n_r / n) * fn(y_right)

def best_split_1d(x, y):
    # Find the threshold on a single feature that maximises Gini reduction.
    thresholds = np.unique(x)
    best_gain, best_t = -1.0, None
    for t in thresholds:
        mask = x <= t
        if mask.sum() == 0 or (~mask).sum() == 0:
            continue
        gain = impurity_reduction(y, y[mask], y[~mask])
        if gain > best_gain:
            best_gain, best_t = gain, t
    return best_t, best_gain

# Apply to pixel 27 (a central pixel) on a binary subset: digit 0 vs digit 1
mask_01 = np.isin(y_tr, [0, 1])
x_demo = X_tr[mask_01, 27]
y_demo = (y_tr[mask_01] == 1).astype(int)

t_star, gain_star = best_split_1d(x_demo, y_demo)
print(f"Best split on pixel 27 (digit 0 vs 1):")
print(f"  threshold = {t_star:.1f},  Gini reduction = {gain_star:.4f}")
print(f"  Left ({(x_demo <= t_star).sum()} pts): digit 0 fraction = "
      f"{(y_demo[x_demo <= t_star] == 0).mean():.3f}")
print(f"  Right ({(x_demo > t_star).sum()} pts): digit 1 fraction = "
      f"{(y_demo[x_demo > t_star] == 1).mean():.3f}")

**In-class exercise.** Repeat the search above with `criterion='entropy'`. Do the two
criteria select the same threshold? Gini and entropy tend to agree on which split to make
but can differ near ties — the main practical difference is computational cost.

---
## Part B — Fitting and visualising a decision tree

We fit a depth-3 tree on the full Digits training set and examine its structure and
decision boundary.

In [ ]:
clf3 = DecisionTreeClassifier(max_depth=3, random_state=0)
clf3.fit(X_tr, y_tr)
print(export_text(clf3, feature_names=[f"px{i}" for i in range(64)]))

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(clf3, feature_names=[f"px{i}" for i in range(64)],
          class_names=[str(d) for d in range(10)],
          filled=True, rounded=True, fontsize=7, ax=ax)
ax.set_title("Decision tree (max_depth=3) — Digits", fontsize=12)
plt.tight_layout(); plt.show()
print(f"Leaves: {clf3.get_n_leaves()}   Depth: {clf3.get_depth()}")
print(f"Train accuracy: {clf3.score(X_tr, y_tr):.4f}")
print(f"Test accuracy:  {clf3.score(X_te, y_te):.4f}")

In [ ]:
feat_a, feat_b = 27, 35
X_tr_2d = X_tr[:, [feat_a, feat_b]]

clf_2d = DecisionTreeClassifier(max_depth=4, random_state=0)
clf_2d.fit(X_tr_2d, y_tr)

h = 0.2
x1_min, x1_max = X_dig[:, feat_a].min() - 0.5, X_dig[:, feat_a].max() + 0.5
x2_min, x2_max = X_dig[:, feat_b].min() - 0.5, X_dig[:, feat_b].max() + 0.5
xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h), np.arange(x2_min, x2_max, h))
Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(xx, yy, Z, cmap="tab10", alpha=0.4)
sc = ax.scatter(X_tr_2d[:, 0], X_tr_2d[:, 1], c=y_tr, cmap="tab10", edgecolors="k", s=14, linewidths=0.3)
ax.set_xlabel(f"Pixel {feat_a}"); ax.set_ylabel(f"Pixel {feat_b}")
ax.set_title("Decision boundary (depth 4, 2 features) — Digits")
plt.colorbar(sc, ax=ax, label="Digit class")
plt.tight_layout(); plt.show()

**In-class exercise.** The decision boundaries above are strictly axis-aligned
rectangles. Explain why this is an intrinsic property of binary decision trees,
regardless of depth.

---
## Part C — Depth, bias–variance, and tree instability

We study how depth controls the bias–variance trade-off, then demonstrate that unpruned
trees are highly sensitive to the training sample — the property that motivates every
ensemble method in Parts F onward.

In [ ]:
depths = list(range(1, 21))
train_err, test_err = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=0)
    clf.fit(X_tr, y_tr)
    train_err.append(1 - clf.score(X_tr, y_tr))
    test_err.append(1 - clf.score(X_te, y_te))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_err, "o-", label="Train error", color="steelblue")
plt.plot(depths, test_err,  "o-", label="Test error",  color="darkorange")
plt.xlabel("max_depth"); plt.ylabel("Misclassification rate")
plt.title("Depth vs error — Digits")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv_mean, cv_se = [], []
for d in depths:
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=d, random_state=0), X_tr, y_tr, cv=skf, scoring="accuracy"
    )
    cv_mean.append(1 - scores.mean())
    cv_se.append(scores.std() / np.sqrt(5))

cv_mean, cv_se = np.array(cv_mean), np.array(cv_se)
best_idx = cv_mean.argmin()
threshold = cv_mean[best_idx] + cv_se[best_idx]
ose_idx = np.where(cv_mean <= threshold)[0][0]

plt.figure(figsize=(9, 4))
plt.plot(depths, cv_mean, "o-", color="steelblue", label="CV error")
plt.fill_between(depths, cv_mean - cv_se, cv_mean + cv_se, alpha=0.2, color="steelblue")
plt.axhline(threshold, color="darkorange", ls="--", label="1-SE threshold")
plt.axvline(depths[ose_idx], color="green", ls=":", label=f"1-SE choice: depth={depths[ose_idx]}")
plt.xlabel("max_depth"); plt.ylabel("CV misclassification rate")
plt.title("5-fold CV with 1-SE rule — Digits")
plt.legend(); plt.tight_layout(); plt.show()
print(f"CV minimiser: depth = {depths[best_idx]}  (error = {cv_mean[best_idx]:.4f})")
print(f"1-SE choice:  depth = {depths[ose_idx]}  (error = {cv_mean[ose_idx]:.4f})")

In [ ]:
# Tree instability: two trees on bootstrap resamples
idx_a = rng.integers(0, len(X_tr), len(X_tr))
idx_b = rng.integers(0, len(X_tr), len(X_tr))

clf_a = DecisionTreeClassifier(max_depth=None, random_state=0)
clf_b = DecisionTreeClassifier(max_depth=None, random_state=1)
clf_a.fit(X_tr[idx_a], y_tr[idx_a])
clf_b.fit(X_tr[idx_b], y_tr[idx_b])

print(f"Tree A root split: pixel {clf_a.tree_.feature[0]}")
print(f"Tree B root split: pixel {clf_b.tree_.feature[0]}")
print(f"Same root feature: {clf_a.tree_.feature[0] == clf_b.tree_.feature[0]}")

agree = (clf_a.predict(X_te) == clf_b.predict(X_te)).mean()
print(f"Agreement on test predictions: {agree:.4f}")
print("Even modest bootstrap variation produces different structures and predictions —")
print("this instability is exactly what Parts F onward turn into an advantage.")

---
## Part D — Cost-complexity pruning

We grow the full unpruned tree, extract its pruning path, and select $\alpha$ by
cross-validation.

In [ ]:
clf_full = DecisionTreeClassifier(random_state=0)
clf_full.fit(X_tr, y_tr)
print(f"Full tree: depth = {clf_full.get_depth()}, leaves = {clf_full.get_n_leaves()}")
print(f"Train error: {1 - clf_full.score(X_tr, y_tr):.4f}")
print(f"Test error:  {1 - clf_full.score(X_te, y_te):.4f}")

path = clf_full.cost_complexity_pruning_path(X_tr, y_tr)
alphas, impurities = path.ccp_alphas, path.impurities
print(f"\nPruning path: {len(alphas)} subtrees, alpha range "
      f"[{alphas[0]:.6f}, {alphas[-1]:.6f}]")

In [ ]:
alpha_candidates = alphas[:-1][alphas[:-1] > 0]
idx_grid = np.round(np.linspace(0, len(alpha_candidates)-1, 30)).astype(int)
alpha_grid = alpha_candidates[idx_grid]

cv_scores = []
for a in alpha_grid:
    scores = cross_val_score(
        DecisionTreeClassifier(ccp_alpha=a, random_state=0), X_tr, y_tr, cv=5, scoring="accuracy"
    )
    cv_scores.append(scores.mean())

best_alpha = alpha_grid[np.argmax(cv_scores)]
clf_pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=0)
clf_pruned.fit(X_tr, y_tr)

print(f"CV-optimal alpha: {best_alpha:.6f}")
print(f"Pruned tree: depth = {clf_pruned.get_depth()}, leaves = {clf_pruned.get_n_leaves()}")
print(f"Full tree test error:   {1 - clf_full.score(X_te, y_te):.4f}")
print(f"Pruned tree test error: {1 - clf_pruned.score(X_te, y_te):.4f}")

**In-class exercise.** For three values of alpha — one well below, one near, and one
well above the CV-optimal alpha — state the number of leaves and the test error. What
pattern do you observe as alpha increases along the path?

---
## Part E — Feature importance: MDI vs permutation, and MDI's cardinality bias

Two measures of variable importance: **MDI** (mean decrease in impurity) is fast but
biased towards high-cardinality features; **permutation importance** measures the drop
in test accuracy when a feature is shuffled, and is not susceptible to that bias. We
compare both on the pruned tree from Part D, then isolate the bias mechanism directly
with a synthetic example.

In [ ]:
mdi = clf_pruned.feature_importances_
feat_names = [f"px{i}" for i in range(64)]

perm = permutation_importance(clf_pruned, X_te, y_te, n_repeats=20, random_state=0, scoring="accuracy")
perm_mean = perm.importances_mean

top10_mdi  = np.argsort(mdi)[::-1][:10]
top10_perm = np.argsort(perm_mean)[::-1][:10]
overlap = len(set(top10_mdi) & set(top10_perm))
print(f"Top-10 overlap between MDI and permutation importance: {overlap}/10")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(mdi.reshape(8, 8), cmap="hot")
axes[0].set_title("MDI by pixel location"); axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(perm_mean.reshape(8, 8), cmap="hot")
axes[1].set_title("Permutation importance by pixel"); axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.suptitle("Feature importance heatmaps — single tree (Digits)", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Synthetic: 5 informative features, one high-cardinality noise feature,
# one low-cardinality noise feature, isolating the MDI cardinality bias directly.
n = 600
X_inf = rng.standard_normal((n, 5))
X_noise_hc = rng.uniform(0, 1, (n, 1)) * 100
X_noise_lc = rng.integers(0, 3, (n, 1)).astype(float)
X_bias = np.hstack([X_inf, X_noise_hc, X_noise_lc])
y_bias = (X_inf[:, 0] + X_inf[:, 1] > 0).astype(int)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_bias, y_bias, test_size=0.3, random_state=0)

clf_bias = DecisionTreeClassifier(max_depth=6, random_state=0)
clf_bias.fit(X_tr_b, y_tr_b)

mdi_bias = clf_bias.feature_importances_
feat_labels = [f"inf_{i}" for i in range(5)] + ["noise_HC", "noise_LC"]
perm_bias = permutation_importance(clf_bias, X_te_b, y_te_b, n_repeats=20, random_state=0).importances_mean

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colours = ["steelblue"]*5 + ["firebrick", "darkorange"]
for ax, scores, title in zip(axes, [mdi_bias, perm_bias], ["MDI", "Permutation importance"]):
    ax.bar(feat_labels, scores, color=colours)
    ax.set_title(title); ax.set_ylabel("Importance")
    ax.set_xticklabels(feat_labels, rotation=30, ha="right")
plt.suptitle("MDI bias: high-cardinality noise inflates MDI (red bar)", y=1.02)
plt.tight_layout(); plt.show()

print("MDI rank of noise_HC (0 = highest):  ", sorted(mdi_bias, reverse=True).index(mdi_bias[5]))
print("Perm rank of noise_HC (0 = highest): ", sorted(perm_bias, reverse=True).index(perm_bias[5]))

MDI assigns inflated importance to `noise_HC` because the tree can always find some
threshold on a continuous feature with many unique values that gives a small impurity
reduction, even with no real signal. Permutation importance is unaffected, since shuffling
a pure-noise feature causes no drop in accuracy regardless of its cardinality.

**In-class exercise.** `noise_LC` (three distinct values) ranks lower than `noise_HC` by
MDI despite both being pure noise. Explain why cardinality, not just informativeness,
drives MDI rankings.

---
## Part F — Bagging: the variance formula and bagging from scratch

The lecture showed $\mathrm{Var}(\bar f) = \rho\sigma^2 + (1-\rho)\sigma^2/B$. We verify
this numerically, then build a bagging classifier from scratch and confirm it beats the
single unstable tree from Part C.

In [ ]:
def simulate_ensemble_variance(B, rho, sigma2=0.25, n_sim=8000, seed=0):
    rng_sim = np.random.default_rng(seed)
    Z = rng_sim.normal(0, np.sqrt(sigma2), n_sim)
    eps = rng_sim.normal(0, np.sqrt(sigma2 * (1 - rho)), (n_sim, B))
    preds = np.sqrt(rho) * Z[:, None] + np.sqrt(1 - rho) * eps
    return preds.mean(axis=1).var()

B_values = [1, 5, 10, 25, 50, 100, 200]
rho_values = [0.0, 0.3, 0.6, 0.9]
sigma2 = 0.25

fig, ax = plt.subplots(figsize=(7, 4))
for rho in rho_values:
    empirical = [simulate_ensemble_variance(B, rho, sigma2) for B in B_values]
    ax.plot(B_values, empirical, marker="o", label=f"rho={rho}")
    ax.axhline(rho * sigma2, color="grey", lw=0.8, ls="--")
ax.set_xlabel("Number of trees B"); ax.set_ylabel("Variance of ensemble mean")
ax.set_title("Ensemble variance vs B for different pairwise correlations")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
class BaggingFromScratch:
    def __init__(self, n_estimators=50, max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.rng = np.random.default_rng(random_state)
        self.trees_ = []

    def fit(self, X, y):
        n = X.shape[0]
        for _ in range(self.n_estimators):
            idx = self.rng.integers(0, n, size=n)
            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X[idx], y[idx])
            self.trees_.append(tree)
        return self

    def predict(self, X):
        votes = np.array([t.predict(X) for t in self.trees_])
        return mode(votes, axis=0).mode.ravel()

bag = BaggingFromScratch(n_estimators=100, max_depth=None, random_state=1)
bag.fit(X_tr, y_tr)
bag_acc = accuracy_score(y_te, bag.predict(X_te))

single_tree = DecisionTreeClassifier(max_depth=None, random_state=1)
single_tree.fit(X_tr, y_tr)
tree_acc = accuracy_score(y_te, single_tree.predict(X_te))

print(f"Single unpruned tree accuracy: {tree_acc:.4f}")
print(f"Bagging (100 trees) accuracy:  {bag_acc:.4f}")
print(f"Improvement: {bag_acc - tree_acc:+.4f}")

**In-class exercise.** Looking at the variance plot, explain in one or two sentences why
increasing $B$ beyond about 50 gives diminishing returns when $\rho$ is large.

---
## Part G — Random Forests: tuning $m$

A Random Forest adds feature subsampling to bagging: at each split, only $m$ features are
considered ($m = \lfloor\sqrt{p}\rfloor = 8$ by default here, since $p=64$). We sweep $m$
directly and check the effect on both OOB and test error.

In [ ]:
p = X_tr.shape[1]
m_values = [1, 2, 4, 8, 16, 32, 64]

oob_by_m, test_by_m = [], []
for m in m_values:
    rf_m = RandomForestClassifier(n_estimators=150, max_features=m, oob_score=True, random_state=4)
    rf_m.fit(X_tr, y_tr)
    oob_by_m.append(1 - rf_m.oob_score_)
    test_by_m.append(1 - accuracy_score(y_te, rf_m.predict(X_te)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(m_values, oob_by_m, marker="o", label="OOB error", color="steelblue")
ax.plot(m_values, test_by_m, marker="s", label="Test error", color="tomato")
ax.axvline(int(np.sqrt(p)), ls=":", color="grey", label=f"sqrt(p) = {int(np.sqrt(p))}")
ax.set_xlabel("m (max_features per split)"); ax.set_ylabel("Error rate")
ax.set_title("RF error vs m (Digits, B=150)")
ax.legend(); plt.tight_layout(); plt.show()

best_m = m_values[np.argmin(oob_by_m)]
print(f"Best m by OOB: {best_m}  (default sqrt(p) = {int(np.sqrt(p))})")

In [ ]:
# Does averaging over many trees change the importance picture from Part E?
rf_imp = RandomForestClassifier(n_estimators=200, random_state=5)
rf_imp.fit(X_tr, y_tr)
mdi_rf = rf_imp.feature_importances_

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(mdi_rf.reshape(8, 8), cmap="hot")
ax.set_title("MDI, averaged over 200 trees"); ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

single_tree_mdi_var = np.var([DecisionTreeClassifier(random_state=s).fit(X_tr, y_tr).feature_importances_
                                for s in range(10)], axis=0).mean()
rf_mdi_var = np.var([RandomForestClassifier(n_estimators=200, random_state=s).fit(X_tr, y_tr).feature_importances_
                       for s in range(3)], axis=0).mean()
print(f"Mean across-run variance of MDI, single trees (10 runs): {single_tree_mdi_var:.2e}")
print(f"Mean across-run variance of MDI, RF of 200 trees (3 runs): {rf_mdi_var:.2e}")
print("Averaging over many trees stabilises the importance estimate, the same mechanism")
print("that stabilises the ensemble prediction itself.")

**In-class exercise.** The OOB error curve as a function of $m$ typically shows a
U-shape. Explain qualitatively why both very small $m$ and $m=p$ (full bagging) lead to
higher error.

---
## Part H — Gradient boosting with early stopping

Gradient boosting builds shallow trees sequentially, each fitting the residuals of the
current ensemble. Unlike bagging and Random Forests, more trees can eventually overfit,
so the learning rate and the number of stages must be chosen together.

In [ ]:
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=6, stratify=y_tr)

learning_rates = [0.5, 0.1, 0.05, 0.01]
colors = ["steelblue", "tomato", "seagreen", "darkorange"]

fig, ax = plt.subplots(figsize=(7, 4))
for lr_val, col in zip(learning_rates, colors):
    gb = GradientBoostingClassifier(n_estimators=300, learning_rate=lr_val, max_depth=3, random_state=7)
    gb.fit(X_tr2, y_tr2)
    val_errors = [1 - accuracy_score(y_val, y_pred) for y_pred in gb.staged_predict(X_val)]
    ax.plot(val_errors, label=f"lr={lr_val}", color=col, lw=1.2)
ax.set_xlabel("Boosting stage"); ax.set_ylabel("Validation error")
ax.set_title("GB: validation error by learning rate (Digits)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
gb_es = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=8)
gb_es.fit(X_tr2, y_tr2)

val_errors_es = [1 - accuracy_score(y_val, y_pred) for y_pred in gb_es.staged_predict(X_val)]
best_stage = int(np.argmin(val_errors_es)) + 1

gb_best = GradientBoostingClassifier(n_estimators=best_stage, learning_rate=0.1, max_depth=3, random_state=8)
gb_best.fit(X_tr, y_tr)

print(f"Optimal stage (by val error):  {best_stage}")
print(f"Validation error at optimum:   {val_errors_es[best_stage - 1]:.4f}")
print(f"Test accuracy (refitted):      {accuracy_score(y_te, gb_best.predict(X_te)):.4f}")

**In-class exercise.** Suppose `learning_rate` increases from 0.1 to 0.5 while
`n_estimators` stays fixed. What happens to the training error, and why might the test
error increase?

---
## Part I — Side-by-side comparison

We train all three ensemble families with comparable budgets and compare test accuracy
and training time.

In [ ]:
models = {
    "Bagging (B=200)": BaggingClassifier(n_estimators=200, random_state=9),
    "Random Forest (B=200)": RandomForestClassifier(n_estimators=200, random_state=9),
    "Gradient Boosting (200 stages)": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=9
    ),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    acc = accuracy_score(y_te, model.predict(X_te))
    results[name] = {"accuracy": acc, "time_s": elapsed}
    print(f"{name:<40} acc={acc:.4f}  time={elapsed:.2f}s")

In [ ]:
names = list(results.keys())
accs  = [results[n]["accuracy"] for n in names]
times = [results[n]["time_s"]   for n in names]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].barh(names, accs, color=["steelblue", "seagreen", "tomato"])
axes[0].set_xlabel("Test accuracy"); axes[0].set_title("Accuracy comparison")
axes[0].set_xlim(0.9, 1.0)
axes[1].barh(names, times, color=["steelblue", "seagreen", "tomato"])
axes[1].set_xlabel("Training time (s)"); axes[1].set_title("Training time comparison")
plt.tight_layout(); plt.show()

**In-class exercise.** Gradient boosting is typically slower to train than a Random
Forest of the same size. Explain why, in terms of the sequential vs parallel structure of
the two methods.

---
## Take-home exercises

**Exercise 1.** Extract the full cost-complexity pruning path from Part D more carefully:
identify all crossover values of alpha (values where the optimal subtree changes), and
plot the number of leaves against alpha on a log scale. For three values of alpha — one
before, one at, and one after the CV-optimal alpha — state the number of leaves and the
test error.

**Exercise 2.** Build a depth-2 tree on a two-class subset of Digits (digits 3 and 8
only). Extract the feature used at each node (`tree_.feature`), the impurity at each node
(`tree_.impurity`), and the sample count (`tree_.n_node_samples`), and compute MDI by
hand for all internal nodes. Verify your result matches `feature_importances_`
(normalised to sum to 1).

**Exercise 3.** The variance formula from Part F was verified on a synthetic Gaussian
model. Verify it on real trees instead: fit $B=50$ decision trees (depth 5) on 50
bootstrap resamples of the Digits training set. At 20 fixed test points, compute the
empirical variance of the 50 predicted class probabilities and the mean pairwise
correlation of the 50 prediction vectors, and compare $\rho\sigma^2 + (1-\rho)\sigma^2/B$
to the empirical variance of the averaged prediction.

**Exercise 4.** Using a grid of `max_depth` values $\{3,5,10,15,\text{None}\}$ and
`max_features` values $\{4,8,16,32\}$, train a Random Forest ($B=150$,
`oob_score=True`) for each combination and record OOB error. Plot a heatmap of OOB
errors, identify the best pair, and confirm the result on the test set.

**Exercise 5.** Fix `n_estimators=200`, `learning_rate=0.1`, and vary `max_depth` over
$\{1,2,3,5,8\}$. Record both training and test accuracy at each depth. At what depth
does overfitting become apparent, and how does this relate to the bias–variance
trade-off from Part C?